In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, f1_score, fbeta_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
training_df = pd.read_csv(
    filepath_or_buffer='training_faults_diagnostics.csv',
    low_memory=False
)

In [3]:
# Convert SPN and FMI values to strings
training_df['spn'] = training_df['spn'].astype(str)
training_df['fmi'] = training_df['fmi'].astype(str)
print(f'spn datatype: {training_df['spn'].dtype}')

spn datatype: object


In [4]:
target = 'Derate_Target'

# Create dataset with features
X = training_df

# Create array of targets
y = training_df[target]

## Identify features for imputing missing values

In [5]:
# Group categorical columns
categorical_columns = ['EquipmentID', 'spn', 'fmi', 'active', 'Severity_Level']

# Group numeric columns
numeric_columns = [
    'BarometricPressure',
    'EngineCoolantTemperature',
    'EngineLoad',
    'EngineOilPressure',
    'EngineOilTemperature',
    'EngineRpm',
    'FuelRate',
    'FuelTemperature',
    'IntakeManifoldTemperature',
    'Speed',
    'Throttle',
    'TurboBoostPressure'
  ]

## Split training dataset

In [22]:
random_state = 1

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.1,
    random_state=random_state,
    stratify=y
)

## Create pipeline and fit model

In [24]:
categorical_pipe = Pipeline(
    steps=[
        ('categorical_imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore'))
    ]
)

numeric_pipe = Pipeline(
    steps=[
        ('numeric_imputer', IterativeImputer(max_iter=5, random_state=random_state)),
        ('scaler', StandardScaler())
    ]
)

In [25]:
ct = ColumnTransformer(
    transformers=[
        ('categorical_pipe', categorical_pipe, categorical_columns),
        ('numeric_pipe', numeric_pipe, numeric_columns)
    ]
)

In [26]:
pipe = Pipeline(
    steps=[
        ('transformer', ct),
        ('model', MLPClassifier(
            hidden_layer_sizes=(32,32,32),
            activation='relu',
            random_state=random_state,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=2
        ))
    ]
)

In [27]:
pipe.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Pipeline(steps=[('transformer',
                 ColumnTransformer(transformers=[('categorical_pipe',
                                                  Pipeline(steps=[('categorical_imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['EquipmentID', 'spn', 'fmi',
                                                   'active',
                                                   'Severity_Level']),
                                                 ('numeric_pipe',
                                                  Pipeline(steps=[('numeric_imputer',
                                                                   IterativeImputer(max_iter=5,
                                                                                    rand...
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['BarometricPressure',
                                                   'EngineCoolantTemperature',
                                                   'EngineLoad',
                                                   'EngineOilPressure',
                                                   'EngineOilTemperature',
                                                   'EngineRpm', 'FuelRate',
                                                   'FuelTemperature',
                                                   'IntakeManifoldTemperature',
                                                   'Speed', 'Throttle',
                                                   'TurboBoostPressure'])])),
                ('model',
                 MLPClassifier(early_stopping=True,
                               hidden_layer_sizes=(32, 32, 32),
                               n_iter_no_change=2, random_state=1))])

## Adjust Threshold

In [28]:
y_pred_prob_val = pipe.predict_proba(X_val)[:,1]

In [29]:
labels=[0, 1, 2]

candidate_thresholds = np.arange(
    start=0.1,
    stop=0.925,
    step=0.01
)

thresholds_df = pd.DataFrame({'threshold': candidate_thresholds})

In [30]:
# fbeta score
thresholds_df['fbeta'] = thresholds_df['threshold'].apply(lambda x: fbeta_score(
    y_true=y_val,
    y_pred=y_pred_prob_val >= x,
    beta=2,
    labels=labels,
    average='macro'
))
thresholds_df.sort_values(by='fbeta', ascending=False).head()

,threshold,fbeta
0,0.10,0.398891
1,0.11,0.386226
2,0.12,0.379338
3,0.13,0.366701
6,0.16,0.365890


In [31]:
# f1 score
thresholds_df['f1'] = thresholds_df['threshold'].apply(lambda x: f1_score(
    y_true=y_val,
    y_pred=y_pred_prob_val >= x,
    labels=labels,
    average='macro'
))
thresholds_df.sort_values(by='f1', ascending=False).head()

,threshold,fbeta,f1
0,0.10,0.398891,0.385994
1,0.11,0.386226,0.378542
2,0.12,0.379338,0.376087
6,0.16,0.365890,0.371094
5,0.15,0.365321,0.369272


## Compare training and testing

In [32]:
threshold = thresholds_df['threshold'].iloc[0]

y_pred_proba_train = pipe.predict_proba(X_train)[:,1]
y_pred_proba_test = pipe.predict_proba(X_test)[:,1]

y_pred_train = y_pred_proba_train >= threshold
y_pred_test = y_pred_proba_test >= threshold

In [33]:
training_cr = classification_report(
    y_true=y_train,
    y_pred=y_pred_train,
    digits=6
)
print(str(training_cr))

training_cm = confusion_matrix(
    y_true=y_train,
    y_pred=y_pred_train,
    labels=labels
)
print(training_cm)

              precision    recall  f1-score   support

           0   0.998855  0.999038  0.998947    950562
           1   0.167213  0.319149  0.219449       799
           2   0.000000  0.000000  0.000000       901

    accuracy                       0.997523    952262
   macro avg   0.388689  0.439396  0.406132    952262
weighted avg   0.997212  0.997523  0.997347    952262

[[949648    914      0]
 [   544    255      0]
 [   545    356      0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [34]:
testing_cr = classification_report(
    y_true=y_test,
    y_pred=y_pred_test,
    digits=6
)
print(str(testing_cr))

testing_cm = confusion_matrix(
    y_true=y_test,
    y_pred=y_pred_test
)
print(testing_cm)

              precision    recall  f1-score   support

           0   0.998836  0.998949  0.998892    211236
           1   0.158192  0.314607  0.210526       178
           2   0.000000  0.000000  0.000000       200

    accuracy                       0.997429    211614
   macro avg   0.385676  0.437852  0.403140    211614
weighted avg   0.997184  0.997429  0.997285    211614

[[211014    222      0]
 [   122     56      0]
 [   124     76      0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Predict Unseen Data

In [35]:
testing_df = pd.read_csv(
    filepath_or_buffer='testing_faults_diagnostics.csv',
    low_memory=False
)

# Convert SPN and FMI values to strings
testing_df['spn'] = testing_df['spn'].astype(str)
testing_df['fmi'] = testing_df['fmi'].astype(str)
print(f'spn datatype: {testing_df['spn'].dtype}')

unseen_data = testing_df.drop(columns=['Derate_Target'])
y_true = testing_df['Derate_Target']

spn datatype: object


In [36]:
y_pred_proba_unseen = pipe.predict_proba(unseen_data)[:,1]

y_pred_unseen = y_pred_proba_unseen >= threshold

In [37]:
unseen_cr = classification_report(
    y_true=y_true,
    y_pred=y_pred_unseen,
    digits=6
)
print(str(unseen_cr))

unseen_cm = confusion_matrix(
    y_true=y_true,
    y_pred=y_pred_unseen
)
print(unseen_cm)

              precision    recall  f1-score   support

           0   0.998497  0.994231  0.996360    128970
           1   0.060213  0.258883  0.097701       197
           2   0.000000  0.000000  0.000000        99

    accuracy                       0.992349    129266
   macro avg   0.352903  0.417705  0.364687    129266
weighted avg   0.996302  0.992349  0.994227    129266

[[128226    744      0]
 [   146     51      0]
 [    47     52      0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
